In [1]:
import pandas as pd
import numpy as np

import matplotlib as plt

In [218]:
df = pd.read_csv("https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv")

In [219]:
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [220]:
df.columns = df.columns.str.lower().str.replace(" ","_")
df.columns

Index(['lead_source', 'industry', 'employment_status', 'location',
       'annual_income', 'number_of_courses_viewed', 'interaction_count',
       'lead_score', 'converted'],
      dtype='object')

In [221]:
df.dtypes

lead_source                  object
industry                     object
employment_status            object
location                     object
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [222]:
categorical = df.dtypes[df.dtypes == 'object'].index
categorical

Index(['lead_source', 'industry', 'employment_status', 'location'], dtype='object')

In [223]:
numerical = df.dtypes[df.dtypes != 'object'].index
numerical

Index(['annual_income', 'number_of_courses_viewed', 'interaction_count',
       'lead_score', 'converted'],
      dtype='object')

In [224]:
for c in categorical:
    df[c] = df[c].str.lower().str.replace(" ","_")
    

In [225]:
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


## Data preparation

In [23]:
# Check if the missing values are presented in the features.
# If there are missing values:
# For categorical features, replace them with 'NA'
# For numerical features, replace with with 0.0

In [226]:
# Check if the missing values are presented in the features.
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [227]:
# For categorical features, replace them with 'NA'
for c in categorical:
    df[c] = df[c].fillna('NA')

In [228]:
# For numerical features, replace them with zeros
for c in numerical:
    df[c] = df[c].fillna(0)

In [229]:
# Check if the missing values filled
df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

## Question 1
#### What is the most frequent observation (mode) for the column 'industry'?

In [42]:
df['industry'].mode()

0    technology
dtype: object

In [43]:
df['industry'].value_counts()

technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: industry, dtype: int64

#### Ans1: Technology

## Question 2
#### Create the correlation matrix for the numerical features of your dataset. In a correlation matrix, you compute the correlation coefficient between every pair of features.What are the two features that have the biggest correlation?

In [47]:
df[numerical]

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,88160.0,3,4,0.64,1
1,72688.0,3,7,0.72,1
2,44697.0,3,4,0.58,1
3,0.0,3,6,0.57,0
4,24062.0,4,5,0.62,1
...,...,...,...,...,...
4995,45834.0,2,2,0.37,1
4996,84831.0,2,5,0.51,0
4997,53636.0,4,6,0.75,1
4998,69430.0,4,8,0.79,1


In [52]:
matrix = df[numerical].corr()
matrix

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
annual_income,1.000000,0.161300,0.122842,0.229496,0.189652
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204,0.395038
interaction_count,0.122842,0.721609,1.000000,0.915746,0.448624
lead_score,0.229496,0.757204,0.915746,1.000000,0.483246
converted,0.189652,0.395038,0.448624,0.483246,1.000000


In [197]:
corr_matrix= matrix.unstack().drop_duplicates()
corr_matrix.sort_values(ascending=False)

annual_income             annual_income               1.000000
interaction_count         lead_score                  0.915746
number_of_courses_viewed  lead_score                  0.757204
                          interaction_count           0.721609
lead_score                converted                   0.483246
interaction_count         converted                   0.448624
number_of_courses_viewed  converted                   0.395038
annual_income             lead_score                  0.229496
                          converted                   0.189652
                          number_of_courses_viewed    0.161300
                          interaction_count           0.122842
dtype: float64

#### Ans 2: interaction_count   and      lead_score                  0.915746 

In [59]:
# Split the data
# Split the data with these exact calls:
# df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
# df_train, df_val = train_test_split(
#     df_full_train, test_size=0.25, random_state=42
# )
# Make sure that the target value converted is not in your dataframe.

In [230]:
from sklearn.model_selection import train_test_split

In [296]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

In [297]:
# Correct the indexes after split
df_full_train = df_full_train.reset_index(drop=True)
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [298]:
# Now extract y from above dfs
y_full_train = df_full_train['converted'].values
y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

In [68]:
del df_full_train['converted']
del df_train['converted']
del df_val['converted']
del df_test['converted']

### Question 3
#### Calculate the mutual information score between converted and other categorical variables in the dataset. Use the training set only.
#### Round the scores to 2 decimals using round(score, 2).
#### Which of these variables has the biggest mutual information score?

In [79]:
 from sklearn.metrics import mutual_info_score

In [81]:
def mutual_info_churn_score(series):
    return mutual_info_score(df_train['converted'], series)

In [86]:
mutual_score = df_train[categorical].apply(mutual_info_churn_score)
mutual_score.sort_values(ascending=False)

lead_source          0.034346
employment_status    0.018966
industry             0.001995
location             0.001043
dtype: float64

#### Ans 3: Lead soruce has highed mutual info score with 'Converted'

### Question 4
#### Now let's train a logistic regression.
#### Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.
#### Fit the model on the training dataset.
#### To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:
#### model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
#### Calculate the accuracy on the validation dataset and round it to 2 decimal digits.
#### What accuracy did you get?

In [92]:
# One hot encoding for categorical features
from sklearn.feature_extraction import DictVectorizer

In [234]:
dv = DictVectorizer(sparse=False)

In [235]:
df_train.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,paid_ads,finance,employed,asia,65980.0,2,3,0.37,1
1,events,finance,employed,south_america,71662.0,0,0,0.09,0
2,referral,NA,employed,asia,40220.0,2,5,0.55,1
3,organic_search,finance,employed,north_america,88352.0,2,2,0.45,0
4,events,technology,employed,north_america,89316.0,1,5,0.00,1


In [236]:
del df_train['converted']

In [237]:
# Convert categorical to a dictionary
dict= df_train.to_dict(orient='records')
dict [0]

{'lead_source': 'paid_ads',
 'industry': 'finance',
 'employment_status': 'employed',
 'location': 'asia',
 'annual_income': 65980.0,
 'number_of_courses_viewed': 2,
 'interaction_count': 3,
 'lead_score': 0.37}

In [238]:
dv.fit(dict)

DictVectorizer(sparse=False)

In [239]:
X_train = dv.transform(dict)

In [240]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)

In [241]:
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

In [242]:
del df_val['converted']

In [143]:
# Prepare X_val to check accuracy

In [243]:
val_dict = df_val.to_dict(orient='records')
dv.fit(val_dict)
X_val = dv.transform(val_dict)
X_val

array([[8.0457e+04, 0.0000e+00, 1.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        3.0000e+00],
       [7.4480e+04, 0.0000e+00, 0.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        0.0000e+00],
       [4.3944e+04, 0.0000e+00, 0.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        2.0000e+00],
       ...,
       [6.2439e+04, 0.0000e+00, 0.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        0.0000e+00],
       [4.0878e+04, 0.0000e+00, 0.0000e+00, ..., 1.0000e+00, 0.0000e+00,
        1.0000e+00],
       [4.8769e+04, 0.0000e+00, 1.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        3.0000e+00]])

In [244]:
# predict using X_val
y_pred = model.predict_proba(X_val)[:,1]

In [245]:
# check accuracy by comparing above prediction with y_val
convert_decision = (y_pred >= 0.5)

In [246]:
original_accuracy = (convert_decision == y_val).mean()
original_accuracy

0.645

In [247]:
original_accuracy.round(2)

0.64

#### Ans:4= 0.65 accuracy approx

### Question 5

In [190]:
original_accuracy

0.645

In [256]:
# lets identify correlation of Numerical and catgorical features with target variable 'Converted'
num_matrix = df_train[numerical].corrwith(df_train['converted'])
num_matrix.sort_values(ascending=False)

converted                   1.000000
lead_score                  0.484635
interaction_count           0.455749
number_of_courses_viewed    0.390132
annual_income               0.177537
dtype: float64

In [257]:
# lets identify correlation of Numerical and catgorical features with target variable 'Converted'
mutual_score = df_train[categorical].apply(mutual_info_churn_score)
mutual_score.sort_values(ascending=False)

lead_source          0.034346
employment_status    0.018966
industry             0.001995
location             0.001043
dtype: float64

In [258]:
## Question has three options:'lead_source', 'number_of_courses_viewed', 'interaction_count'

#### Accuracy without: lead_source

In [261]:
df_train.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,paid_ads,finance,employed,asia,65980.0,2,3,0.37,1
1,events,finance,employed,south_america,71662.0,0,0,0.09,0
2,referral,NA,employed,asia,40220.0,2,5,0.55,1
3,organic_search,finance,employed,north_america,88352.0,2,2,0.45,0
4,events,technology,employed,north_america,89316.0,1,5,0.00,1


In [299]:
del df_train['lead_source']

In [301]:
del df_train['converted']

In [302]:
df_train.head(2)

,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score
0,finance,employed,asia,65980.0,2,3,0.37
1,finance,employed,south_america,71662.0,0,0,0.09


In [305]:
dict= df_train.to_dict(orient='records')

from sklearn.feature_extraction import DictVectorizer
dv = DictVectorizer(sparse=False)
dv.fit(dict)
X_train = dv.transform(dict)

In [306]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)

model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

In [307]:
del df_val['converted']
del df_val['lead_source']

In [308]:
val_dict = df_val.to_dict(orient='records')
dv.fit(val_dict)
X_val = dv.transform(val_dict)
# predict using X_val
y_pred = model.predict_proba(X_val)[:,1]

In [311]:
convert_decision_without_lead_source = (y_pred >= 0.5)


In [312]:
accuracy_without_lead_source = (convert_decision_without_lead_source == y_val).mean()
accuracy_without_lead_source

0.642

In [368]:
diff1 = original_accuracy - accuracy_without_lead_source
diff1

0.0030000000000000027

#### Difference between original and without lead source = 0.00300

### Accuracy without: number_of_courses_viewed

In [329]:
exec(In[296])

In [330]:
exec(In[297])

In [331]:
exec(In[298])

In [332]:
del df_train['converted']
del df_train['number_of_courses_viewed']

del df_val['converted']
del df_val['number_of_courses_viewed']

In [333]:
df_train.head()

,lead_source,industry,employment_status,location,annual_income,interaction_count,lead_score
0,paid_ads,finance,employed,asia,65980.0,3,0.37
1,events,finance,employed,south_america,71662.0,0,0.09
2,referral,NA,employed,asia,40220.0,5,0.55
3,organic_search,finance,employed,north_america,88352.0,2,0.45
4,events,technology,employed,north_america,89316.0,5,0.00


In [334]:
dict= df_train.to_dict(orient='records')

from sklearn.feature_extraction import DictVectorizer
dv = DictVectorizer(sparse=False)
dv.fit(dict)
X_train = dv.transform(dict)

In [335]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)

model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

In [336]:
val_dict = df_val.to_dict(orient='records')
dv.fit(val_dict)
X_val = dv.transform(val_dict)
# predict using X_val
y_pred = model.predict_proba(X_val)[:,1]

In [338]:
convert_decision_without_no_cources_viewed = (y_pred >= 0.5)

In [343]:
accuracy_without_no_courses_viewed = (convert_decision_without_no_cources_viewed == y_val).mean()
accuracy_without_no_courses_viewed

0.643

In [367]:
diff2 = original_accuracy - accuracy_without_no_courses_viewed
diff2 

0.0020000000000000018

#### Difference between original and without No courses viewed = 0.00200

### Calculate Accuracy without: interaction_count

In [351]:
exec(In[296])

In [352]:
exec(In[297])

In [353]:
exec(In[298])

In [354]:
del df_train['converted']
del df_train['interaction_count']

del df_val['converted']
del df_val['interaction_count']

In [355]:
df_train.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,lead_score
0,paid_ads,finance,employed,asia,65980.0,2,0.37
1,events,finance,employed,south_america,71662.0,0,0.09
2,referral,NA,employed,asia,40220.0,2,0.55
3,organic_search,finance,employed,north_america,88352.0,2,0.45
4,events,technology,employed,north_america,89316.0,1,0.00


In [356]:
dict= df_train.to_dict(orient='records')

from sklearn.feature_extraction import DictVectorizer
dv = DictVectorizer(sparse=False)
dv.fit(dict)
X_train = dv.transform(dict)

In [357]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)

model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

In [358]:
val_dict = df_val.to_dict(orient='records')
dv.fit(val_dict)
X_val = dv.transform(val_dict)
# predict using X_val
y_pred = model.predict_proba(X_val)[:,1]

In [359]:
convert_decision_without_interaction_count = (y_pred >= 0.5)

In [360]:
accuracy_without_interaction_count = (convert_decision_without_interaction_count == y_val).mean()
accuracy_without_interaction_count

0.601

In [366]:
diff3 = original_accuracy - accuracy_without_interaction_count
diff3

0.04400000000000004

#### Difference between original and without interaction_count = 0.04400

In [374]:
diff1, diff2, diff3

(0.0030000000000000027, 0.0020000000000000018, 0.04400000000000004)

## Ans:5 no_courses_viewed has least difference with original accuracy i.e. 0.0020000

## Question 6

In [384]:
exec(In[296])

In [385]:
exec(In[297])

In [386]:
exec(In[298])

In [387]:
del df_train['converted']
del df_val['converted']

In [388]:
dict= df_train.to_dict(orient='records')

from sklearn.feature_extraction import DictVectorizer
dv = DictVectorizer(sparse=False)
dv.fit(dict)
X_train = dv.transform(dict)

#### train with c=0.000001

In [ ]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(solver='liblinear', C=0.000001, max_iter=1000, random_state=42)

model.fit(X_train, y_train)

val_dict = df_val.to_dict(orient='records')
dv.fit(val_dict)
X_val = dv.transform(val_dict)
# predict using X_val
y_pred = model.predict_proba(X_val)[:,1]

convert_decision = (y_pred >= 0.5)
accuracy_c1 = (convert_decision == y_val).mean()
accuracy_c1

In [394]:
def train_model(c_parm):
    from sklearn.linear_model import LogisticRegression
    model = LogisticRegression(solver='liblinear', C=c_parm, max_iter=1000, random_state=42)

    model.fit(X_train, y_train)

    val_dict = df_val.to_dict(orient='records')
    dv.fit(val_dict)
    X_val = dv.transform(val_dict)
    # predict using X_val
    y_pred = model.predict_proba(X_val)[:,1]

    convert_decision = (y_pred >= 0.5)
    accuracy_c1 = (convert_decision == y_val).mean()
    
    return accuracy_c1.round(3)

In [395]:
accuracy_c1 = train_model(0.000001)
accuracy_c1

0.598

In [396]:
accuracy_c2 = train_model(0.00001)
accuracy_c2

0.598

In [398]:
accuracy_c3 = train_model(0.0001)
accuracy_c3

0.613

In [399]:
accuracy_c4 = train_model(0.001)
accuracy_c4

0.645

### Ans 6: c=0.001 has max accuracy